# NB-R10 — Results Compilation for Revised Manuscript
**Purpose:** Load all outputs from NB-R01 through NB-R09 and compile the updated tables and figures needed for the revised manuscript. Provides a single source of truth for all numbers cited in the response-to-reviewers letter.

**Outputs produced:**
- Table 1: Clean split summary (R3.1 fix)
- Table 2: Feature engineering specification (R2.6)
- Table 3: Hyperparameter table (R1.7, R3.4)
- Table 4: Regime-stratified accuracy with class-stratified metrics (R1.6, R3.6)
- Table 5: Statistical tests (block bootstrap, block permutation, non-overlapping Fisher) (R1.2, R2.2, R3.3)
- Table 6: Baseline comparison (R2.3)
- Table 7: Ablation results (R3.4)
- Table 8: Trading simulation corrected (R1.4, R1.5, R2.4)
- Table 9: Walk-forward generalizability (R1.3, R2.2)
- Table 10: VIX threshold comparison (R1.1, R2.1, R3.2)

In [1]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from pathlib import Path

PROJ    = Path(r'F:\MLSAPU\PhD-SPPU\India-VIX-Major-Revision')
PROC    = PROJ / 'data' / 'processed'
RESULTS = PROJ / 'results'
PLOTS   = PROJ / 'plots'

print('Loading all result files...')

# Load all results
split_summary   = pd.read_csv(PROC / 'split_summary.csv')
hp_table        = pd.read_csv(RESULTS / 'hyperparameter_table.csv')
stats           = json.load(open(RESULTS / 'statistical_tests.json'))
baselines       = pd.read_csv(RESULTS / 'baseline_comparison.csv')
ablation        = pd.read_csv(RESULTS / 'ablation_results.csv')
trading         = pd.read_csv(RESULTS / 'trading_simulation.csv')
walkforward     = pd.read_csv(RESULTS / 'walkforward_results.csv')
vix_cfg         = json.load(open(RESULTS / 'vix_threshold_config.json'))
threshold_comp  = pd.read_csv(RESULTS / 'threshold_comparison.csv')

print('[OK] All results loaded.')

Loading all result files...
[OK] All results loaded.


## Table 1: Clean Data Splits

In [2]:
print('TABLE 1: Clean Data Splits (fixes R3.1 -- label leakage at split boundaries)')
print(split_summary.to_string(index=False))

TABLE 1: Clean Data Splits (fixes R3.1 -- label leakage at split boundaries)
Split      Start End (clean)  Rows Up% (21d)                             Note
Train 2016-05-23  2023-09-11  1794     62.0% Last 21 days of boundary removed
  Val 2023-10-20  2024-12-04   273     64.8% Last 21 days of boundary removed
 Test 2025-01-14  2026-02-25   276     68.8%          NaN-label rows excluded


## Table 2: Feature Engineering Specification (R2.6)

In [3]:
feat_spec = pd.DataFrame([
    {'Feature': 'MACD',          'Formula/Params': 'EMA(12) - EMA(26); EMAs use exponential weighting, adjust=False',         'Leakage-safe': 'Yes'},
    {'Feature': 'MACD Signal',   'Formula/Params': 'EMA(9) of MACD',                                                           'Leakage-safe': 'Yes'},
    {'Feature': 'MACD Histogram','Formula/Params': 'MACD - MACD Signal',                                                        'Leakage-safe': 'Yes'},
    {'Feature': 'EMA-20',        'Formula/Params': 'Exponential MA, span=20, adjust=False',                                     'Leakage-safe': 'Yes'},
    {'Feature': 'RSI-14',        'Formula/Params': '100 - 100/(1+AvgGain/AvgLoss); Wilder smoothing (com=13)',                  'Leakage-safe': 'Yes'},
    {'Feature': 'Stoch %K',      'Formula/Params': '100*(Close-Low14)/(High14-Low14); 14-day lookback',                         'Leakage-safe': 'Yes'},
    {'Feature': 'Stoch %D',      'Formula/Params': '3-day SMA of %K',                                                           'Leakage-safe': 'Yes'},
    {'Feature': 'ROC-10',        'Formula/Params': '(Close/Close[t-10] - 1) * 100; 10-day rate of change',                      'Leakage-safe': 'Yes'},
    {'Feature': 'BB Upper',      'Formula/Params': 'SMA(20) + 2*std(20); simple MA, 20-day window',                             'Leakage-safe': 'Yes'},
    {'Feature': 'BB Lower',      'Formula/Params': 'SMA(20) - 2*std(20)',                                                        'Leakage-safe': 'Yes'},
    {'Feature': 'BB Width',      'Formula/Params': '(BB_Upper - BB_Lower) / SMA(20)',                                            'Leakage-safe': 'Yes'},
    {'Feature': 'ATR-14',        'Formula/Params': 'Wilder EMA(com=13) of True Range; TR=max(H-L, |H-C[t-1]|, |L-C[t-1]|)',    'Leakage-safe': 'Yes'},
    {'Feature': 'Log Ret Lag 1', 'Formula/Params': 'log(Close[t] / Close[t-1])',                                                 'Leakage-safe': 'Yes'},
    {'Feature': 'Log Ret Lag 2', 'Formula/Params': 'log(Close[t] / Close[t-2])',                                                 'Leakage-safe': 'Yes'},
    {'Feature': 'Log Ret Lag 3', 'Formula/Params': 'log(Close[t] / Close[t-3])',                                                 'Leakage-safe': 'Yes'},
    {'Feature': 'Log Ret Lag 5', 'Formula/Params': 'log(Close[t] / Close[t-5])',                                                 'Leakage-safe': 'Yes'},
])
print('TABLE 2: Feature Engineering Specification')
print(feat_spec.to_string(index=False))
feat_spec.to_csv(RESULTS / 'feature_specification.csv', index=False)

TABLE 2: Feature Engineering Specification
       Feature                                                        Formula/Params Leakage-safe
          MACD       EMA(12) - EMA(26); EMAs use exponential weighting, adjust=False          Yes
   MACD Signal                                                        EMA(9) of MACD          Yes
MACD Histogram                                                    MACD - MACD Signal          Yes
        EMA-20                                 Exponential MA, span=20, adjust=False          Yes
        RSI-14              100 - 100/(1+AvgGain/AvgLoss); Wilder smoothing (com=13)          Yes
      Stoch %K                     100*(Close-Low14)/(High14-Low14); 14-day lookback          Yes
      Stoch %D                                                       3-day SMA of %K          Yes
        ROC-10                  (Close/Close[t-10] - 1) * 100; 10-day rate of change          Yes
      BB Upper                         SMA(20) + 2*std(20); simple MA, 20-d

## Table 3: Hyperparameters

In [4]:
print('TABLE 3: Final Hyperparameters (from Optuna, 50 trials, TimeSeriesSplit n=5)')
print(hp_table.to_string(index=False))

TABLE 3: Final Hyperparameters (from Optuna, 50 trials, TimeSeriesSplit n=5)
              Model                                                                                                                                                                                                                           Key Parameters  CV Loss
            XGBoost               {'n_estimators': 176, 'max_depth': 4, 'learning_rate': 0.011632560008838805, 'subsample': 0.773657142746063, 'colsample_bytree': 0.7753603608719395, 'reg_alpha': 0.00022888035398378767, 'reg_lambda': 4.893261502637006}   0.7358
           LightGBM {'n_estimators': 130, 'max_depth': 4, 'learning_rate': 0.010039856303508456, 'num_leaves': 39, 'subsample': 0.9441554065487753, 'colsample_bytree': 0.7421465905117742, 'reg_alpha': 4.043192940864163, 'reg_lambda': 5.855735803974474}   0.6955
      Random Forest                                                                                                                      

## Table 4: Regime-Stratified Accuracy (with Class Metrics)

In [5]:
print('TABLE 4: Regime-Stratified Accuracy and Class-Stratified Metrics (Stacking Ensemble)')
regime_summary = {
    'Metric': ['N', 'Up% (local)', 'Majority-class baseline (%)', 'Accuracy (%)',
               'Precision (%)', 'Recall (%)', 'F1 (%)', 'ROC-AUC'],
    'High-VIX': [
        stats['high_vix']['n'],
        f"{stats['high_vix']['up_pct']:.1f}%",
        f"{stats['high_vix']['majority_baseline_acc']:.1f}%",
        f"{stats['high_vix']['accuracy']:.1f}%",
        f"{stats['high_vix']['precision']:.1f}%",
        f"{stats['high_vix']['recall']:.1f}%",
        f"{stats['high_vix']['f1']:.1f}%",
        f"{stats['high_vix']['auc']:.4f}",
    ],
    'Low-VIX': [
        stats['low_vix']['n'],
        f"{stats['low_vix']['up_pct']:.1f}%",
        f"{stats['low_vix']['majority_baseline_acc']:.1f}%",
        f"{stats['low_vix']['accuracy']:.1f}%",
        f"{stats['low_vix']['precision']:.1f}%",
        f"{stats['low_vix']['recall']:.1f}%",
        f"{stats['low_vix']['f1']:.1f}%",
        f"{stats['low_vix']['auc']:.4f}",
    ]
}
t4 = pd.DataFrame(regime_summary)
print(t4.to_string(index=False))
t4.to_csv(RESULTS / 'table4_regime_metrics.csv', index=False)

TABLE 4: Regime-Stratified Accuracy and Class-Stratified Metrics (Stacking Ensemble)
                     Metric High-VIX Low-VIX
                          N        8     248
                Up% (local)   100.0%   70.2%
Majority-class baseline (%)   100.0%   70.2%
               Accuracy (%)   100.0%   66.5%
              Precision (%)   100.0%   69.4%
                 Recall (%)   100.0%   93.7%
                     F1 (%)   100.0%   79.7%
                    ROC-AUC      nan  0.5838


## Table 5: Statistical Tests Summary

In [6]:
t5 = pd.DataFrame([
    {'Test': 'Accuracy difference (High - Low)',      'Value': f"{stats['accuracy_diff_pp']:.1f} pp", 'Note': ''},
    {'Test': 'Fisher exact (non-overlapping, every 21st day)', 'Value': f"{stats['fisher_non_overlap_p']:.3e}", 'Note': 'Avoids serial correlation'},
    {'Test': 'Fisher Bonferroni-corrected (x3)',      'Value': f"{stats['fisher_bonferroni_p']:.3e}", 'Note': '3 horizons tested'},
    {'Test': 'Block bootstrap 95% CI (block=21)',     'Value': f"[{stats['block_boot_ci_95'][0]:.1f}, {stats['block_boot_ci_95'][1]:.1f}] pp", 'Note': 'Circular block bootstrap'},
    {'Test': 'Block permutation p-value',             'Value': f"{stats['block_perm_p']:.3e}", 'Note': 'Block shuffle of regime labels'},
    {'Test': 'Block permutation Bonferroni (x3)',     'Value': f"{stats['block_perm_bonferroni_p']:.3e}", 'Note': ''},
])
print('TABLE 5: Statistical Tests (Corrected for Overlapping Observations)')
print(t5.to_string(index=False))
t5.to_csv(RESULTS / 'table5_statistical_tests.csv', index=False)

TABLE 5: Statistical Tests (Corrected for Overlapping Observations)
                                          Test           Value                           Note
              Accuracy difference (High - Low)         33.5 pp                               
Fisher exact (non-overlapping, every 21st day)       1.000e+00      Avoids serial correlation
              Fisher Bonferroni-corrected (x3)       1.000e+00              3 horizons tested
             Block bootstrap 95% CI (block=21) [16.2, 51.4] pp       Circular block bootstrap
                     Block permutation p-value       3.566e-01 Block shuffle of regime labels
             Block permutation Bonferroni (x3)       1.000e+00                               


## Table 8: Corrected Trading Simulation

In [8]:
print('TABLE 8: Corrected Trading Simulation Results')
print(f'Sharpe convention: (mean excess daily return / std) * sqrt({252})')
print('Risk-free rate: 6.5% p.a. (India 91-day T-bill, FY2025 avg = 6.5%)')
print()
print(trading.to_string(index=False))

# Highlight the corrected finding
for _, row in trading.iterrows():
    if row['cost_bps'] == 50:
        status = 'above' if row['strat_above_bh'] else 'BELOW (corrected from original manuscript)'
        print(f"\n[CORRECTION] At 50bps: strategy Sharpe={row['strat_sharpe']:.3f} is {status} BH={row['bh_sharpe']:.3f}")

TABLE 8: Corrected Trading Simulation Results
Sharpe convention: (mean excess daily return / std) * sqrt(252)
Risk-free rate: 6.5% p.a. (India 91-day T-bill, FY2025 avg = 6.5%)

 cost_bps  n_trades  deployed_days  strat_sharpe  bh_sharpe  strat_cum_ret_pct  bh_cum_ret_pct  strat_max_dd_pct  bh_max_dd_pct  strat_above_bh
        0         1             21       -11.512      1.101              11.08           19.84               0.0          -6.62           False
       10         1             21       -11.512      1.101              11.08           19.84               0.0          -6.62           False
       20         1             21       -11.512      1.101              11.08           19.84               0.0          -6.62           False
       30         1             21       -11.513      1.101              11.08           19.84               0.0          -6.62           False
       50         1             21       -11.513      1.101              11.08           19.84        

## Final Summary: Reviewer Comments vs Fixes Applied

In [9]:
summary = [
    {'Reviewer': 'R1', 'Comment': 'VIX threshold look-ahead bias',         'Fix': 'NB-R02: p75 of train+val VIX',             'Status': '[DONE]'},
    {'Reviewer': 'R1', 'Comment': 'Overlapping observations / serial corr','Fix': 'NB-R04: block bootstrap + non-overlap Fisher','Status': '[DONE]'},
    {'Reviewer': 'R1', 'Comment': 'Single High-VIX episode generalizability','Fix': 'NB-R05: annual walk-forward validation',   'Status': '[DONE]'},
    {'Reviewer': 'R1', 'Comment': 'Factual error at 50bps cost',            'Fix': 'NB-R09: corrected Table 8 + text',         'Status': '[DONE]'},
    {'Reviewer': 'R1', 'Comment': 'Signal timing ambiguity',                'Fix': 'NB-R09: explicit execution protocol stated','Status': '[DONE]'},
    {'Reviewer': 'R1', 'Comment': 'Class-stratified metrics for High-VIX',  'Fix': 'NB-R04: precision/recall/F1 per regime',   'Status': '[DONE]'},
    {'Reviewer': 'R1', 'Comment': 'Hyperparameter table + wrong DOIs',      'Fix': 'NB-R03: hp table; DOIs fixed manually',    'Status': '[DONE]'},
    {'Reviewer': 'R2', 'Comment': 'VIX threshold look-ahead bias',          'Fix': 'NB-R02 (same as R1)',                      'Status': '[DONE]'},
    {'Reviewer': 'R2', 'Comment': 'Temporal concentration of High-VIX',     'Fix': 'NB-R05 (same as R1)',                      'Status': '[DONE]'},
    {'Reviewer': 'R2', 'Comment': 'Missing simple baseline comparison',      'Fix': 'NB-R06: logistic reg, persistence, maj',  'Status': '[DONE]'},
    {'Reviewer': 'R2', 'Comment': 'Sharpe convention not stated',            'Fix': 'NB-R09: RFR=6.5%, 252 days',              'Status': '[DONE]'},
    {'Reviewer': 'R2', 'Comment': 'SHAP values not actually shown',         'Fix': 'NB-R07: regime-specific SHAP + plots',     'Status': '[DONE]'},
    {'Reviewer': 'R2', 'Comment': 'Feature engineering details missing',     'Fix': 'NB-R10 Table 2 + manuscript Section 3',   'Status': '[DONE]'},
    {'Reviewer': 'R3', 'Comment': 'Label leakage at split boundaries',       'Fix': 'NB-R01: last 21 days trimmed per split',  'Status': '[DONE]'},
    {'Reviewer': 'R3', 'Comment': 'VIX threshold look-ahead bias',          'Fix': 'NB-R02 (same)',                            'Status': '[DONE]'},
    {'Reviewer': 'R3', 'Comment': 'Overlapping / clustering statistics',     'Fix': 'NB-R04 (same as R1)',                      'Status': '[DONE]'},
    {'Reviewer': 'R3', 'Comment': 'Ablation studies',                        'Fix': 'NB-R08: per-model + no-attention ablation','Status': '[DONE]'},
    {'Reviewer': 'R3', 'Comment': 'Old references',                          'Fix': 'Manuscript: added 5 recent refs (2022-25)','Status': '[MANUSCRIPT]'},
    {'Reviewer': 'R3', 'Comment': 'Regime-specific baselines + target date', 'Fix': 'NB-R04 + NB-R10; clarified in manuscript','Status': '[DONE]'},
]

fix_df = pd.DataFrame(summary)
print(fix_df.to_string(index=False))
fix_df.to_csv(RESULTS / 'revision_tracker.csv', index=False)
print('\nRevision tracker saved to results/revision_tracker.csv')

Reviewer                                  Comment                                          Fix       Status
      R1            VIX threshold look-ahead bias                 NB-R02: p75 of train+val VIX       [DONE]
      R1   Overlapping observations / serial corr NB-R04: block bootstrap + non-overlap Fisher       [DONE]
      R1 Single High-VIX episode generalizability       NB-R05: annual walk-forward validation       [DONE]
      R1              Factual error at 50bps cost             NB-R09: corrected Table 8 + text       [DONE]
      R1                  Signal timing ambiguity   NB-R09: explicit execution protocol stated       [DONE]
      R1    Class-stratified metrics for High-VIX       NB-R04: precision/recall/F1 per regime       [DONE]
      R1        Hyperparameter table + wrong DOIs        NB-R03: hp table; DOIs fixed manually       [DONE]
      R2            VIX threshold look-ahead bias                          NB-R02 (same as R1)       [DONE]
      R2       Temporal conc